In [2]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [4]:
!kaggle datasets download -d tmdb/tmdb-movie-metadata

Dataset URL: https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata
License(s): other
  0% 0.00/8.89M [00:00<?, ?B/s]
100% 8.89M/8.89M [00:00<00:00, 1.05GB/s]


In [5]:
! unzip /content/tmdb-movie-metadata.zip

Archive:  /content/tmdb-movie-metadata.zip
  inflating: tmdb_5000_credits.csv   
  inflating: tmdb_5000_movies.csv    


In [6]:
movies = pd.read_csv("/content/tmdb_5000_movies.csv")
credit = pd.read_csv("/content/tmdb_5000_credits.csv")

In [7]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [8]:
credit.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [9]:
print(movies.columns)
print(credit.columns)
# as one column have movie id and one has id we will use it to merge

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')
Index(['movie_id', 'title', 'cast', 'crew'], dtype='object')


In [10]:
merged_df = movies.merge(credit, left_on='id', right_on='movie_id')

merged_df = merged_df.drop(columns=['movie_id'])


In [11]:
merged_df.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title_x,vote_average,vote_count,title_y,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [12]:
columns_to_drop = [
    "homepage",
    "title_x",
    "title_y",
    "status",
    "production_countries",
    "spoken_languages",
    "revenue",
    "release_date",
    "runtime",
    "vote_average",
    "vote_count",
    "budget",
    "tagline"
]

merged_df = merged_df.drop(columns=columns_to_drop)


In [13]:
merged_df.isna().sum()

,0
genres,0
id,0
keywords,0
original_language,0
original_title,0
overview,3
popularity,0
production_companies,0
cast,0
crew,0


In [14]:
# drop missing values
merged_df.dropna(inplace=True)

# Preprocessing


In [15]:
import ast
import pandas as pd

def extract_names(obj, key="name", top_n=None, job_filter=None):
    # if empty
    if pd.isna(obj):
        return []

    try:
        # ast converts a string that looks like a Python object into an actual Python object.
        # "[{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}]"  this is not list  we need to convert it to list
        data = ast.literal_eval(obj)
    except:
        return []

    results = []

    for item in data:
        if job_filter:
            if item.get("job") == job_filter:
                results.append(item.get(key))
                break
        else:
            results.append(item.get(key))

        if top_n and len(results) >= top_n:
            break

    return results



merged_df["genres"] = merged_df["genres"].apply(extract_names)
merged_df["keywords"] = merged_df["keywords"].apply(extract_names)
merged_df["cast"] = merged_df["cast"].apply(lambda x: extract_names(x, top_n=3)) # we have to fetch top 3 cast as they are important
merged_df["crew"] = merged_df["crew"].apply(lambda x: extract_names(x, job_filter="Director"))  # we have to fetch directer name


In [16]:
merged_df["overview"] = merged_df["overview"].fillna("").apply(lambda x: x.split())
# remove spaces between words
for col in ["genres", "keywords", "cast", "crew"]:
    merged_df[col] = merged_df[col].apply(lambda x: [i.replace(" ", "") for i in x])


In [17]:
merged_df["tags"] = (
    merged_df["overview"] +
    merged_df["genres"] +
    merged_df["keywords"] +
    merged_df["cast"] +
    merged_df["crew"]
)


In [18]:
merged_df.head(2)

,genres,id,keywords,original_language,original_title,overview,popularity,production_companies,cast,crew,tags
0,"[Action, Adventure, Fantasy, ScienceFiction]",19995,"[cultureclash, future, spacewar, spacecolony, ...",en,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,"[Adventure, Fantasy, Action]",285,"[ocean, drugabuse, exoticisland, eastindiatrad...",en,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],"[Captain, Barbossa,, long, believed, to, be, d..."


In [19]:
final_df = merged_df[["id","original_title","tags"]]

In [20]:
final_df.head()

,id,original_title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


In [21]:
 # conver the tags list to ==> str
final_df["tags"] = final_df["tags"].apply(lambda x:" ".join(x))

In [22]:
final_df.head()

,id,original_title,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca..."


In [23]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

nltk.download('punkt_tab')
nltk.download('punkt')


stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=6000,stop_words="english")




def preprocessing(text):

    if not isinstance(text,str):
        return ""

    #Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    #Remove special characters & punctuation
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    #lowercase
    text = text.lower()

    # Tokenization
    words = word_tokenize(text)

    #remove stopwords
    words = [word for word in words if word not in stop_words]

    #lemmatization
    words = [lemmatizer.lemmatize(word) for word in words]

    # join back into sentence
    words = " ".join(words)

    return words



final_df["tags"] = final_df["tags"].apply(preprocessing)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


# Vectorization

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=6000)

vectors = tfidf.fit_transform(final_df["tags"]).toarray()


In [25]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)


In [47]:
def recommend(movie,top_k=4):
    movie = movie.lower()

    if movie not in final_df["original_title"].str.lower().values:
        return "Movie not Found"

    # fetch the given movie index from data
    index = final_df[final_df["original_title"].str.lower()==movie].index[0]

    distances = similarity[index] # ==> [1.0, 0.23, 0.87, 0.12, 0.56, ...]


    # [50, 10, 40, 20] ==> sort [10, 20, 40, 50]
    # argsort ==> [1,3,2,0]
    #[::-1]  revers as argsort give most similar movie at last(small--> large)
    movie_indices = distances.argsort()[::-1][1:top_k+1]


    return final_df.iloc[movie_indices]["original_title"].tolist()

In [48]:
recommend("Avatar")


['Aliens', 'Apollo 18', 'Meet Dave', 'Star Trek Into Darkness']